In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_PATH = r"D:\RAC-AI Projects\Khmer-Text-Full-System-Rith\NLP-python_service\models\khmer-mBART50-news-summarization_v1"

print("Loading model...")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False)

# Load model
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_PATH, dtype=torch.float16 if device == "cuda" else torch.float32
)

model.to(device)
model.eval()

print("Model loaded successfully!")


Loading model...
Device: cuda


W0404 10:39:30.515000 43836 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Model loaded successfully!


In [2]:
# Warmup once
warmup_inputs = tokenizer(
    "នេះគឺជាការធ្វើ warmup", return_tensors="pt", truncation=True, max_length=32
)
warmup_inputs = {k: v.to(device) for k, v in warmup_inputs.items()}

with torch.inference_mode():
    _ = model.generate(**warmup_inputs, max_length=12, min_length=5, use_cache=True)


def summarize(text: str, num_summaries: int = 1):
    text = text.strip()
    if not text:
        return {"summaries": [""]}

    try:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_length=200,
                min_length=80,
                do_sample=True,
                top_k=30,
                top_p=0.95,
                temperature=0.85,
                repetition_penalty=1.1,
                num_return_sequences=num_summaries,
                no_repeat_ngram_size=3,
                early_stopping=True,
                use_cache=True,
            )

        summaries = []
        for out in outputs:
            summary = tokenizer.decode(out, skip_special_tokens=True).strip()
            if summary and not summary.endswith("។"):
                summary += "។"
            summaries.append(summary)

        if num_summaries == 1:
            return {"summary": summaries[0]}
        return {"summaries": summaries}

    except Exception as e:
        return {"error": str(e)}

In [3]:
text_to_summarize = """
ប្រធានាធិបតីអាមេរិក លោក ដូណាល់ ត្រាំ បានគិតគូរជាយូរណាស់មកហើយចង់ឱ្យសហរដ្ឋអាមេរិកគ្រប់គ្រងដែនដី Greenland ជាគំនិតមួយដែលធ្វើឱ្យបណ្ដាមេដឹកនាំអឺរ៉ុប និងប្រជាជន Greenland ខឹងសម្បារផង និងសើចចំអកផង។ ប៉ុន្តែនៅបន្ទាប់ពីសហរដ្ឋអាមេរិកវាយប្រហារលើវ៉េណេស៊ុយអេឡា និងចាប់ខ្លួនលោក នីកូឡាស់ ម៉ាឌូរ៉ូ កាលពីថ្ងៃទី៣ ខែមករា វាបានបង្ហាញថាមហិច្ឆតារបស់លោក ត្រាំ ចង់គ្រប់គ្រងដែនដីក្នុងតំបន់អាក់ទិកមួយនេះហាក់មានភាពប្រាកដប្រជាជាងពេលណាៗទាំងអស់។
Greenland ដែលជាតំបន់ស្វយ័តមួយរបស់ដាណឺម៉ាក និងមានប្រជាជនរស់នៅ ៥៧០០០នាក់ បានទទួលឱ្យសហរដ្ឋអាមេរិកដាក់មូលដ្ឋានទ័ពមួយរួចទៅហើយ។ តែរដ្ឋបាលលោក ត្រាំចង់បាន សិទិ្ធអំណាចគ្រប់គ្រងបន្ថែមលើដែនដីមួយនេះ ដោយសារតែ Greenland មានទីភូមិសាស្ត្រយុទ្ធសាស្ត្រយ៉ាងសំខាន់នៅក្នុងតំបន់អាក់ទិកដើម្បីការពារផលប្រយោជន៍សន្តិសុខសហរដ្ឋអាមេរិក និង NATO។ «យើងត្រូវការ Greenland សម្រាប់ការការពារ»។ នេះជាអ្វីដែលលោក ត្រាំ បានប្រាប់កាសែតអាមេរិក The Atlantic កាលពីថ្ងៃទី៤ ខែមករា ជាមួយការអះអាងថា Greenland កំពុងត្រូវបានហ៊ុមព័ទ្ធដោយកប៉ាល់ចិន និងរុស្ស៉ី។ Greenland ក៏សម្បូរផងដែរធនធានធម្មជាតិ ក្នុងនោះរួមមានឧស្ម័នធម្មជាតិ ប្រេង និងរ៉ែដែលត្រូវបានប្រើក្នុងបច្ចេកវិទ្យា និងយោធាដែលកំពុង ក្លាយជាចំណុចក្ដៅមួយក្នុងសង្រ្គាមពាណិជ្ជកម្មរវាងសហរដ្ឋអាមេរិក និងចិន។
តែទោះជា
"""
data = summarize(text_to_summarize, num_summaries=1)
print("Summary:\n", data["summary"])

Summary:
 ប្រធានាធិបតីអាមេរិក លោក ដូណាល់ ត្រាំ បានគិតគូរជាយូរណាស់មកហើយចង់ឱ្យសហរដ្ឋអាមេរិកគ្រប់គ្រងដែនដី Greenland ជាគំនិតមួយ ដែលធ្វើឱ្យបណ្ដាមេដឹកនាំអឺរ៉ុប និងប្រជាជន Greenland ខឹងសម្បារផង និងសើចចំអកផង។ ប៉ុន្តែនៅបន្ទាប់ពីសហរដ្ឋអាមេរិកវាយប្រហារលើវ៉េណេស៊ុយអេឡា និងចាប់ខ្លួនលោក នីកូឡាស់ ម៉ាឌូរ៉ូ កាលពីថ្ងៃទី៣ ខែមករា វាបានបង្ហាញថាមហិច្ឆតារបស់លោក ត្រាំ ចង់គ្រប់គ្រងបន្ថែមលើ Greenland គឺជា គំនិត ដែល មានភាពប្រាកដប្រជាជាងពេលណាៗទាំងអស់។ Greenland ដែលជា។
